# Практическая работа №1
## Дерево решений: классификация сортов изюма

**Цель работы** — применение дерева решений для определения сорта изюма по его геометрическим характеристикам.

В работе был рассмотрен набор данных Raisin с двумя сортами: Kecimen и Besni. Были выполнены проверка и подготовка данных, изучены распределения признаков и различия между классами. Затем было обучено дерево решений и проверено качество классификации. Отдельно были рассмотрены структура дерева, матрица ошибок и важность признаков.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scipy scikit-learn

Для работы были выбраны библиотеки `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy` и `scikit-learn`. Команда `%pip install` устанавливает их в среду, в которой открыт ноутбук. Параметр `-q` уменьшает количество служебных сообщений при установке.

После запуска ячейки могут появиться сообщения об установке пакетов. Числовых результатов анализа на данном этапе ещё нет: библиотеки только подготавливаются к дальнейшей работе.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

# В браузерной среде используется open_url, в локальной — pandas.
try:
    from pyodide.http import open_url
except ImportError:
    open_url = None

from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_curve)
from sklearn.calibration import calibration_curve


Были подключены инструменты для обработки таблиц, вычислений, визуализации и обучения моделей. Для работы с файлами используется `Path`, таблицами — `pandas`, массивами — `numpy`. Графики строятся средствами `matplotlib` и `seaborn`. Из `scikit-learn` импортированы алгоритмы, функции разделения выборки, преобразования признаков и метрики качества. `scipy.stats` применяется для статистических графиков.

Отдельный блок `try / except ImportError` проверяет возможность загрузки файлов через `pyodide.http.open_url`. Это сделано для совместимости с браузерным Jupyter. В обычном Python переменная `open_url` остаётся равной `None`. Если все импорты выполнены без ошибок, ячейка ничего не печатает, но все указанные функции становятся доступны далее.

In [ ]:
def read_csv_compat(url, local_path=None, **read_csv_kwargs):
    if local_path is not None:
        local_path = Path(local_path)
        if local_path.exists():
            return pd.read_csv(local_path, **read_csv_kwargs)

    if open_url is not None:
        data = pd.read_csv(open_url(url), **read_csv_kwargs)
    else:
        data = pd.read_csv(url, **read_csv_kwargs)

    if local_path is not None:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(local_path, index=False)

    return data


Была задана функция `read_csv_compat`, которая используется при загрузке всех четырёх наборов данных. Сначала проверяется наличие локального CSV. Если он уже сохранён, таблица читается с диска. При отсутствии файла выполняется загрузка по URL: в браузерной среде через `open_url`, в обычном Jupyter через `pd.read_csv`. После загрузки данные записываются в папку `data`.

Аргумент `**read_csv_kwargs` позволяет при необходимости передавать дополнительные настройки чтения CSV. Команда `return data` возвращает подготовленную таблицу `DataFrame`. На экране после объявления функции ничего не появляется: фактическое чтение будет выполнено при её вызове.

## 1. Загрузка данных

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

data_path = DATA_DIR / "Raisin_Dataset.csv"
data_url = "https://raw.githubusercontent.com/cynthiakoskei/Raisin-DataSet/main/Raisin_Dataset.csv"

df = read_csv_compat(data_url, data_path)
df.head()


Был указан адрес исходного CSV и путь к локальной копии. Выражение `DATA_DIR.mkdir(exist_ok=True)` создаёт папку `data`, если её ещё нет. Функция `read_csv_compat` возвращает таблицу в переменную `df`. Завершающая команда `df.head()` показывает пять первых строк.

В выводе отображаются названия столбцов и значения первых пяти объектов. По этой таблице проверяется правильность загрузки: не съехали ли столбцы, как представлены признаки и где находится целевая переменная. Это первичный осмотр данных, а не результат работы алгоритма.

## 2. Первичный анализ

In [ ]:
print("Размер датасета:", df.shape)
display(df.head())
df.info()

Для первичной проверки были использованы `df.shape`, `df.head()` и `df.info()`. Первая команда показывает размер набора данных в виде `(число строк, число столбцов)`. Вторая выводит небольшой фрагмент таблицы. Третья показывает тип данных каждого столбца и количество непустых значений (`Non-Null Count`).

Если для какого-либо признака число непустых значений меньше общего числа строк, в нём имеются пропуски. Также по `dtype` можно проверить, что числовые признаки действительно распознаны как числа, а названия категорий — как текст.

In [ ]:
df.describe(include='all').T

Была получена описательная статистика с помощью `describe()`. Показатель `count` отражает число заполненных значений; `mean` — среднее; `std` — стандартное отклонение; `min` и `max` — границы наблюдаемого диапазона. Квартили `25%`, `50%` и `75%` характеризуют положение основной части данных, при этом `50%` соответствует медиане. Операция `.T` разворачивает таблицу, чтобы признаки располагались строками.

По полученной таблице удобно сравнивать диапазоны значений. Большая разница между средним и медианой может говорить об асимметрии распределения, а крайние минимумы и максимумы требуют дополнительной проверки.

## 3. Проверка качества данных

In [ ]:
print("Пропуски:")
display(df.isnull().sum().to_frame("missing"))

print("Количество полных дубликатов:", df.duplicated().sum())
print("\nКоличество объектов по сортам:")
display(df["Class"].value_counts())

Проверка пропусков была выполнена выражением `df.isnull().sum()`: оно подсчитывает отсутствующие значения для каждого столбца. Отдельно командой `df.duplicated().sum()` определяется число полностью совпадающих строк. Таблица `missing` содержит количество пропусков, а следующая строка вывода — число дубликатов.

Нулевые значения в этих результатах означают, что соответствующая проблема не обнаружена. Ненулевые значения показывают, какие столбцы или строки требуют обработки. Для задач классификации дополнительно выводится количество объектов каждого класса: это позволяет оценить их представленность ещё до обучения.

In [ ]:
feature_columns = [
    "Area", "MajorAxisLength", "MinorAxisLength",
    "Eccentricity", "ConvexArea", "Extent", "Perimeter"
]

outliers = {}
for column in feature_columns:
    values = df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outliers[column] = ((values < low) | (values > high)).sum()

pd.Series(outliers, name="Количество значений за границами IQR")

Для семи геометрических характеристик изюма были рассчитаны потенциальные выбросы по IQR. В список `feature_columns` включены площадь, длины осей, эксцентриситет, выпуклая площадь, степень заполнения и периметр. В цикле вычисляются квартили и число наблюдений, выходящих за пределы `1.5 × IQR` от коробки распределения.

В результате выводится таблица по признакам. Большое число потенциальных выбросов может быть связано как с особенностями формы зерна, так и с ошибкой измерения. По одному этому подсчёту строки не удалялись.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
axes = axes.ravel()

for index, column in enumerate(feature_columns):
    axes[index].boxplot(df[column].dropna())
    axes[index].set_title(column)

for index in range(len(feature_columns), len(axes)):
    axes[index].axis("off")

plt.suptitle("Диаграммы размаха признаков", y=1.02)
plt.tight_layout()
plt.show()

Каждый boxplot описывает распределение одного геометрического признака изюма.

Он помогает увидеть крайние значения и проверить, нет ли очевидных ошибок. Выбросы не удаляются автоматически, потому что зерна изюма действительно могут иметь необычные размеры или форму.

In [ ]:
before = len(df)
df = df.drop_duplicates().dropna().copy()

print("Строк до очистки:", before)
print("Строк после очистки:", len(df))

Из таблицы были удалены полные дубликаты и строки с пропущенными значениями. Сравнение `before` и `len(df)` показывает, сколько записей осталось после обработки. Для дальнейших графиков и обучения применяется именно очищенная таблица.

В выводе появляются два размера: до удаления и после него. Если числа совпадают, повторяющихся или неполных записей по применённым правилам не было.

## 4. Разведочный анализ

In [ ]:
df[feature_columns].hist(figsize=(14, 10), bins=20)
plt.suptitle("Распределения признаков", y=1.02)
plt.tight_layout()
plt.show()

Гистограммы показывают, как распределены площадь, длины осей, эксцентриситет, периметр и другие параметры зерен.

Так можно увидеть диапазоны признаков, форму распределений и возможные группы объектов ещё до обучения дерева.

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Class", color=sns.color_palette()[0])
plt.title("Количество объектов каждого сорта")
plt.show()

Столбцы показывают количество объектов сортов `Kecimen` и `Besni`.

Проверяем баланс классов. Если один сорт был бы представлен намного чаще второго, одной Accuracy было бы недостаточно для оценки модели.

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(
    df[feature_columns].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Матрица корреляции признаков")
plt.show()

Матрица отражает линейную связь между геометрическими признаками.

**Важно.** Само дерево решений не требует, чтобы признаки были линейно связаны с целью, но корреляционная матрица всё равно полезна как часть EDA: она показывает, какие характеристики несут похожую информацию.

### Scatter plot: площадь и периметр изюма по сортам

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Area", y="Perimeter", hue="Class", alpha=0.65, s=23)
plt.xlabel("Площадь зерна (пиксели²)")
plt.ylabel("Периметр зерна (пиксели)")
plt.title("Scatter plot: площадь и периметр изюма по сортам")
plt.tight_layout()
plt.show()

Была построена диаграмма рассеяния с цветовой кодировкой классов. `hue="Class"` раскрашивает каждый объект по реальному сорту.

X — площадь зерна в пикселях², Y — периметр в пикселях; одна точка — один образец. Два цвета показывают сорт `Kecimen` или `Besni`.

Если цветовые облака заметно перекрываются, по одной паре признаков сорта нельзя разделить полностью. Дерево анализирует несколько признаков одновременно.

### Violin plot: эксцентриситет по сортам

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x="Class", y="Eccentricity", inner="quartile", cut=0, color=sns.color_palette()[0])
plt.xlabel("Сорт изюма")
plt.ylabel("Эксцентриситет (безразмерный)")
plt.title("Violin plot: эксцентриситет по сортам")
plt.tight_layout()
plt.show()

Была построена скрипичная диаграмма. `violinplot` визуализирует плотность эксцентриситета отдельно для двух сортов и показывает их квартильные линии.

X — сорт, Y — эксцентриситет, отражающий вытянутость аппроксимирующего эллипса. У широкого участка силуэта больше относительная частота значений.

Различные формы и медианы указывают на различия между сортами по вытянутости зерна, но не обязательно дают полное разделение.

### KDE: периметр зерна по сортам

In [ ]:
plt.figure(figsize=(9, 5))
for j, (name, part) in enumerate(df.groupby("Class")):
    sns.kdeplot(data=part, x="Perimeter", label=name, color=sns.color_palette()[j])
plt.xlabel("Периметр зерна, пиксели")
plt.ylabel("Оценка плотности распределения")
plt.title("KDE: периметр зерна по сортам")
plt.legend()
plt.tight_layout()
plt.show()

Для сравнения использован график: ядерная оценка плотности (*KDE*). `groupby("Class")` разделяет выборку, `kdeplot` отображает сглаженные плотности периметра.

X — периметр зерна, Y — относительная плотность. Кривые нормированы отдельно и не показывают абсолютное количество объектов.

Участки, где кривые разделяются, могут быть информативны для классификации. Перекрывающиеся области означают, что один признак неоднозначен.

### Групповой boxplot: параметр Extent по сортам

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Class", y="Extent", color=sns.color_palette()[0])
plt.xlabel("Сорт изюма")
plt.ylabel("Extent: доля площади зерна в ограничивающем прямоугольнике")
plt.title("Групповой boxplot: параметр Extent по сортам")
plt.tight_layout()
plt.show()

Были построены групповые диаграммы размаха (*grouped boxplots*). `sns.boxplot` строит коробку и «усы» для параметра `Extent` каждого сорта.

X — сорт, Y — безразмерная доля занятой площади; линия внутри коробки — медиана, крайние точки — потенциальные выбросы.

Можно сравнить центральные значения и разброс. Если коробки существенно перекрываются, по одному `Extent` классы точно отделить нельзя.

## 5. Подготовка данных

In [ ]:
encoder = LabelEncoder()

X = df[feature_columns]
y = encoder.fit_transform(df["Class"])

print("Кодирование классов:")
for value, code_value in zip(encoder.classes_, encoder.transform(encoder.classes_)):
    print(f"{value} -> {code_value}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Для обучения дерева были сформированы признаки `X` и целевые метки `y`. `LabelEncoder().fit_transform(df["Class"])` переводит два текстовых названия сортов в числа. Цикл с `zip` печатает соответствие между названием и меткой, что понадобится при чтении матрицы ошибок и отчёта классификации.

Данные разделены в пропорции 80/20 с `stratify=y`. В конце выводятся соответствия классов и размеры Train/Test. В обеих частях число признаков должно совпадать.

## 6. Обучение

In [ ]:
model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy на train: {model.score(X_train, y_train):.3f}")
print(f"Accuracy на test:  {model.score(X_test, y_test):.3f}")

Было обучено дерево решений `DecisionTreeClassifier`. Ограничение `max_depth=4` не позволяет дереву расти глубже четырёх уровней, а `min_samples_leaf=5` исключает листья менее чем с пятью обучающими объектами. Эти параметры уменьшают риск подгонки под отдельные записи. `fit()` строит условия разбиения по признакам, `predict()` определяет классы тестовых объектов.

В выводе сравниваются Accuracy на Train и Test. Главное значение для оценки — тестовое: большой разрыв в пользу обучающей части может указывать на переобучение.

## 7. Метрики

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print()
print(
    classification_report(
        y_test,
        y_pred,
        target_names=encoder.classes_,
        digits=3
    )
)

После обучения были подсчитаны Accuracy, Precision, Recall и F1. Они оценивают долю верных решений и качество распознавания положительного класса. `classification_report` рассчитывает показатели отдельно для обоих сортов; через `target_names=encoder.classes_` числовые метки заменяются названиями.

В выводе сначала появляются общие метрики, затем таблица `precision`, `recall`, `f1-score`, `support`. Разница показателей между сортами помогает определить, какой класс модель распознаёт хуже.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)
plt.xlabel("Предсказанный класс")
plt.ylabel("Фактический класс")
plt.title("Матрица ошибок")
plt.show()

Матрица ошибок показывает, сколько объектов каждого сорта классифицировано правильно и сколько перепутано с другим сортом.

Значения на главной диагонали — правильные ответы. Вне диагонали — ошибки классификации.

## 8. Визуализация дерева

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(
    model,
    feature_names=feature_columns,
    class_names=encoder.classes_,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Дерево решений")
plt.show()

Начинаем с верхнего узла. В каждом узле записано условие вида `признак <= порог`. Если условие выполняется, идём по левой ветви, если нет — по правой. В конечном листе находится итоговый класс.

В узлах также можно увидеть количество объектов и степень смешения классов. Чем ближе лист к одному чистому классу, тем увереннее соответствующее решение.

**Зачем этот график нужен.** Это главное преимущество дерева решений: можно проследить, почему модель отнесла объект к конкретному сорту.

От корневого узла можно перейти к листу по последовательности условий. Ограничение глубины делает схему компактнее и снижает риск запоминания отдельных записей.

### Нормированная матрица ошибок дерева (по фактическому классу)

In [ ]:
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(7, 5))
sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Greens", vmin=0, vmax=1,
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Предсказанный сорт")
plt.ylabel("Фактический сорт")
plt.title("Нормированная матрица ошибок дерева (по фактическому классу)")
plt.tight_layout()
plt.show()

Была построена нормированная матрица ошибок (*normalized confusion matrix*). `normalize="true"` переводит число объектов в проценты внутри каждого фактического класса.

Строка соответствует истинному сорту, столбец — прогнозу. Диагональ показывает полноту распознавания (Recall) соответствующего сорта.

Проценты позволяют объективно сопоставлять ошибки сортов, даже если число тестовых объектов по сортам немного различается.

### Precision, Recall и F1 дерева решений по сортам

In [ ]:
report_tree = classification_report(y_test, y_pred, target_names=encoder.classes_,
                                    output_dict=True, zero_division=0)
positions = np.arange(len(encoder.classes_))
width = 0.25
plt.figure(figsize=(9, 5))
for j, metric in enumerate(["precision", "recall", "f1-score"]):
    plt.bar(positions + (j-1)*width,
            [report_tree[name][metric] for name in encoder.classes_], width, label=metric)
plt.xticks(positions, encoder.classes_)
plt.ylim(0, 1.05)
plt.ylabel("Значение метрики [0, 1]")
plt.title("Precision, Recall и F1 дерева решений по сортам")
plt.legend()
plt.tight_layout()
plt.show()

Для сравнения использован график: сгруппированная столбчатая диаграмма качества по классам. `classification_report(..., output_dict=True)` предоставляет метрики для обоих сортов, а `plt.bar` сравнивает их на одной шкале.

X — сорт изюма, Y — значение Precision, Recall или F1. Три соседних столбца описывают один сорт.

Отличие Recall между сортами указывает на неравномерную способность дерева обнаруживать их; это не видно из одной общей Accuracy.

### ROC-кривая дерева решений (класс 1)

In [ ]:
class_one_prob = model.predict_proba(X_test)[:, 1]
fpr_tree, tpr_tree, _ = roc_curve(y_test, class_one_prob)
auc_tree = roc_auc_score(y_test, class_one_prob)
plt.figure(figsize=(8, 5))
plt.plot(fpr_tree, tpr_tree, marker="o", label=f"ROC-AUC = {auc_tree:.3f}")
plt.plot([0, 1], [0, 1], ls="--", color="gray", label="Случайное угадывание")
plt.xlabel("Доля ложноположительных (FPR)")
plt.ylabel("Полнота положительного класса (TPR)")
plt.title("ROC-кривая дерева решений (класс 1)")
plt.legend()
plt.tight_layout()
plt.show()

Для сравнения использован график: ROC-кривая классификатора дерева. `predict_proba` извлекает вероятность класса `1`, затем `roc_curve` и `roc_auc_score` строят кривую и вычисляют площадь.

X — FPR, Y — TPR при разных порогах, пунктир — случайное угадывание.

Ближе к верхнему левому углу — лучше. У дерева вероятности обычно принимают ограниченное число значений по листьям, поэтому ROC может выглядеть ступенчатой; это ожидаемо.

### Кривая сложности модели: глубина и точность дерева

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
depths = np.arange(1, 9)
cv_scores = []
train_scores = []
for depth in depths:
    candidate = DecisionTreeClassifier(max_depth=int(depth), min_samples_leaf=5, random_state=42)
    cv_scores.append(cross_val_score(candidate, X_train, y_train, cv=cv, scoring="accuracy").mean())
    candidate.fit(X_train, y_train)
    train_scores.append(candidate.score(X_train, y_train))
plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores, marker="o", label="Accuracy на Train")
plt.plot(depths, cv_scores, marker="s", label="Средняя CV Accuracy на Train")
plt.xlabel("Максимальная глубина дерева")
plt.ylabel("Accuracy [0, 1]")
plt.title("Кривая сложности модели: глубина и точность дерева")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Для сравнения использован график: диагностическая кривая сложности модели (*model complexity curve*). Для глубин 1–8 считается Accuracy на всех обучающих данных и средняя четырёхкратная кросс-валидация **исключительно внутри Train**.

X — максимальная глубина, Y — Accuracy; две линии показывают качество на Train и среднее качество на CV-фолдах.

Увеличение Train Accuracy при застое/падении CV Accuracy может сигнализировать о переобучении. **Тестовая выборка здесь не используется для выбора глубины**; она остаётся независимой от диагностической проверки.

### Распределение заполненности конечных листьев дерева

In [ ]:
leaf_ids = model.apply(X_train)
leaf_sizes = pd.Series(leaf_ids).value_counts().to_numpy()
plt.figure(figsize=(8, 5))
plt.hist(leaf_sizes, bins=min(12, len(leaf_sizes)), edgecolor="black", alpha=0.75)
plt.xlabel("Число обучающих образцов в конечном листе")
plt.ylabel("Число листьев дерева")
plt.title("Распределение заполненности конечных листьев дерева")
plt.tight_layout()
plt.show()

Для сравнения использован график: гистограмма размеров листьев дерева (*leaf occupancy histogram*). `model.apply(X_train)` возвращает номер листа для каждого обучающего объекта; `value_counts()` показывает, сколько объектов в каждом листе.

X — число наблюдений на лист, Y — сколько листьев имеет такой размер.

Множество маленьких листьев указывает на более дробные правила, которые могут быть чувствительны к шуму. У текущей модели размеры не должны быть меньше `min_samples_leaf=5` на обучающей выборке.

In [ ]:
importance = pd.DataFrame({
    "Признак": feature_columns,
    "Важность": model.feature_importances_
}).sort_values("Важность", ascending=False)

importance

Для каждого геометрического признака была получена оценка `model.feature_importances_`. Значения характеризуют его вклад в уменьшение неоднородности при разбиениях дерева. Таблица `importance` сортируется по убыванию, после чего показывается результат.

Показатель, близкий к нулю, означает, что данный признак почти не влиял на разбиения; значение ровно 0 — что он не использовался в построенном дереве. Сумма важностей всех признаков равна единице, если дерево выполняло разбиения.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance,
    x="Важность",
    y="Признак",
    color=sns.color_palette()[0]
)
plt.title("Важность признаков дерева решений")
plt.show()

Чем длиннее столбец, тем больше соответствующий признак использовался деревом для полезных разделений данных.

Признак с важностью `0` фактически не участвовал в итоговых решениях дерева. Сумма всех важностей равна примерно `1`.

## Вывод

Дерево решений использовано для определения сорта изюма по геометрическим признакам. Главное преимущество модели в этой задаче — наглядность: результат можно представить в виде последовательности условий. Ограничение глубины уменьшает переобучение. Сравнение Accuracy на обучающей и тестовой выборках позволяет дополнительно проверить, не запомнила ли модель обучающие данные слишком сильно.